# Fine-tuning Qwen3-4B for General Medical QA with Unsloth

**Project:** Intelligent Medical Assistant for Healthcare Consultation  
**Notebook:** `01-train-qwen3-medical-qa.ipynb`  
**Model:** `unsloth/Qwen3-4B-unsloth-bnb-4bit`  
**Dataset:** `TVNPeter/medical-data`

---

## Table of Contents

1. [Notebook Goal](#1-notebook-goal)  
2. [Kaggle Runtime Setup](#2-kaggle-runtime-setup)  
3. [Install Dependencies](#3-install-dependencies)  
4. [Import Libraries](#4-import-libraries)  
5. [Configuration](#5-configuration)  
6. [Load Dataset from Hugging Face](#6-load-dataset-from-hugging-face)  
7. [Inspect Dataset Structure](#7-inspect-dataset-structure)  
8. [Filter General Medical QA Categories](#8-filter-general-medical-qa-categories)  
9. [Prepare Chat Training Format](#9-prepare-chat-training-format)  
10. [Load Qwen3-4B 4-bit with Unsloth](#10-load-qwen3-4b-4-bit-with-unsloth)  
11. [Attach LoRA Adapters](#11-attach-lora-adapters)  
12. [Training Setup](#12-training-setup)  
13. [Train the Model](#13-train-the-model)  
14. [Run Inference Test](#14-run-inference-test)  
15. [Save LoRA Adapter](#15-save-lora-adapter)  
16. [Optional: Push Adapter to Hugging Face](#16-optional-push-adapter-to-hugging-face)  
17. [LangGraph QA Node Skeleton](#17-langgraph-qa-node-skeleton)  
18. [Next Step](#18-next-step)

<a id="1-notebook-goal"></a>
## 1. Notebook Goal

This notebook fine-tunes **Qwen3-4B** for the first branch of the system:

```text
User Question
      ↓
General Medical QA Chatbot
      ↓
Answer
```

At this stage, we only train the **general QA branch**.  
The **medical risk / RAG branch** will be built later and connected through LangGraph.

<a id="2-kaggle-runtime-setup"></a>
## 2. Kaggle Runtime Setup

Recommended Kaggle settings:

```text
Accelerator: GPU T4 x2 or P100
Internet: ON
Persistence: Files only if needed
```

For 4-bit Qwen3-4B, **T4 x2** is usually better than CPU-only.  
If Kaggle gives CUDA memory errors, reduce:

```python
MAX_SEQ_LENGTH
PER_DEVICE_TRAIN_BATCH_SIZE
GRADIENT_ACCUMULATION_STEPS
```

<a id="3-install-dependencies"></a>
## 3. Install Dependencies

Run this cell once at the beginning of the Kaggle notebook.

In [1]:
%%capture
!pip install -U unsloth
!pip install -U transformers datasets accelerate bitsandbytes peft trl
!pip install -U huggingface_hub

<a id="4-import-libraries"></a>
## 4. Import Libraries

In [2]:
import os
import torch
import pandas as pd
from collections import Counter
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


<a id="5-configuration"></a>
## 5. Configuration

Change these values if you want a larger/smaller experiment.

In [3]:
MODEL_NAME = "unsloth/Qwen3-4B-unsloth-bnb-4bit"
DATASET_NAME = "TVNPeter/medical-data"

OUTPUT_DIR = "/kaggle/working/qwen3-4b-medical-qa-lora"
MERGED_OUTPUT_DIR = "/kaggle/working/qwen3-4b-medical-qa-merged"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# Training config
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
WARMUP_STEPS = 50
LOGGING_STEPS = 10
SAVE_STEPS = 500

# Use a smaller subset for debugging. Set to None for full training.
DEBUG_TRAIN_SAMPLES = None
DEBUG_EVAL_SAMPLES = None

SEED = 42

<a id="6-load-dataset-from-hugging-face"></a>
## 6. Load Dataset from Hugging Face

In [4]:
ds = load_dataset(DATASET_NAME)
ds

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/473M [00:00<?, ?B/s]

eval.jsonl:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/63977 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3367 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
        num_rows: 63977
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
        num_rows: 3367
    })
})

<a id="7-inspect-dataset-structure"></a>
## 7. Inspect Dataset Structure

In [5]:
print(ds)
print("\nTrain columns:", ds["train"].column_names)
print("Test columns:", ds["test"].column_names)

print("\nFirst sample:")
ds["train"][0]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
        num_rows: 63977
    })
    test: Dataset({
        features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
        num_rows: 3367
    })
})

Train columns: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count']
Test columns: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count']

First sample:


{'id': 'auto_f5370a6e0f4f45a5863edf31e0da5a93d770b2a5',
 'question': 'Một bệnh nhân đang dùng Calcigenol và Orlistat (thuốc giảm hấp thu lipid) xảy ra triệu chứng chuột rút cơ do hạ calci máu. Đâu là cơ chế tương tác thuốc và cách quản lý rủi ro này?',
 'answer': '**Cơ chế tương tác thuốc** Tình huống bệnh nhân của bạn gặp phải là một tương tác dược lý học có cơ sở rõ ràng dựa trên cơ chế dược động học của Orlistat và Vitamin D (thành phần trong Calcigenol): 1. **Cơ chế ức chế hấp thu Lipid**: Orlistat là thuốc ức chế lipase tuyến tụy, ngăn cản quá trình thủy phân chất béo trung tính thành axit béo tự do và monoglyceride, từ đó làm giảm khoảng 30% lượng chất béo hấp thu từ thức ăn. 2. **Ảnh hưởng đến Vitamin D tan trong mỡ**: Vitamin D là một vitamin tan trong mỡ. Quá trình hấp thu Vitamin D vào cơ thể phụ thuộc vào nhũ tương hóa và hòa tan trong các giọt lipid trong ống tiêu hóa. Khi Orlistat làm giảm hấp thu lipid, nó gián tiếp làm giảm đáng kể khả năng hấp thu Vitamin D từ chế độ ăn

In [6]:
print("Category distribution in train:")
train_counter = Counter(ds["train"]["category"])
for cat, count in train_counter.most_common():
    print(f"{cat:25s} {count}")

print("\nCategory distribution in test:")
test_counter = Counter(ds["test"]["category"])
for cat, count in test_counter.most_common():
    print(f"{cat:25s} {count}")

Category distribution in train:
factual                   7759
mechanism                 7514
usage                     6682
safety                    6427
indications               6300
inquiry                   6201
comparison                2789
interactions              1749
pharmacokinetics          1746
side_effects              1743
missed_dose               1742
overdose                  1703
contraindications         1671
pregnancy                 1532
case_based                1143
usage_application         1139
identification            1119
edge_case                 1113
trad_vs_modern            1105
contraindication          1085
patient_query             1048
pediatric                 647
geriatric                 11
lifestyle                 7
preparation               2

Category distribution in test:
factual                   405
mechanism                 385
indications               353
safety                    332
usage                     332
inquiry             

<a id="8-filter-general-medical-qa-categories"></a>
## 8. Filter General Medical QA Categories

For the first stage, we train the model only on lower-risk general QA categories.

The high-risk categories such as `overdose`, `pregnancy`, `pediatric`, `interactions`, `contraindications`, `patient_query`, `case_based`, and `edge_case` will be handled later by the RAG branch.

In [7]:
GENERAL_CATEGORIES = [
    "factual",
    "mechanism",
    "usage",
    "indications",
    "inquiry",
    "comparison",
    "identification",
    "pharmacokinetics",
    "trad_vs_modern",
]

RISK_CATEGORIES = [
    "safety",
    "interactions",
    "contraindications",
    "contraindication",
    "pregnancy",
    "overdose",
    "pediatric",
    "patient_query",
    "case_based",
    "edge_case",
]

print("General categories:", GENERAL_CATEGORIES)
print("Risk categories:", RISK_CATEGORIES)

General categories: ['factual', 'mechanism', 'usage', 'indications', 'inquiry', 'comparison', 'identification', 'pharmacokinetics', 'trad_vs_modern']
Risk categories: ['safety', 'interactions', 'contraindications', 'contraindication', 'pregnancy', 'overdose', 'pediatric', 'patient_query', 'case_based', 'edge_case']


In [8]:
train_general = ds["train"].filter(lambda x: x["category"] in GENERAL_CATEGORIES)
test_general = ds["test"].filter(lambda x: x["category"] in GENERAL_CATEGORIES)

if DEBUG_TRAIN_SAMPLES is not None:
    train_general = train_general.select(range(min(DEBUG_TRAIN_SAMPLES, len(train_general))))

if DEBUG_EVAL_SAMPLES is not None:
    test_general = test_general.select(range(min(DEBUG_EVAL_SAMPLES, len(test_general))))

print(train_general)
print(test_general)

print("\nFiltered train category distribution:")
print(Counter(train_general["category"]))

Filter:   0%|          | 0/63977 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3367 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
    num_rows: 41215
})
Dataset({
    features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
    num_rows: 2168
})

Filtered train category distribution:
Counter({'factual': 7759, 'mechanism': 7514, 'usage': 6682, 'indications': 6300, 'inquiry': 6201, 'comparison': 2789, 'pharmacokinetics': 1746, 'identification': 1119, 'trad_vs_modern': 1105})


<a id="9-prepare-chat-training-format"></a>
## 9. Prepare Chat Training Format

The dataset already contains a `messages` field:

```python
[
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."}
]
```

We convert it into the chat template format expected by Qwen.

In [9]:
def validate_messages(example):
    messages = example.get("messages", None)
    if not isinstance(messages, list) or len(messages) < 2:
        return False
    if messages[0].get("role") != "user":
        return False
    if messages[-1].get("role") != "assistant":
        return False
    if not messages[0].get("content") or not messages[-1].get("content"):
        return False
    return True

train_general = train_general.filter(validate_messages)
test_general = test_general.filter(validate_messages)

print(train_general)
print(test_general)

Filter:   0%|          | 0/41215 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2168 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
    num_rows: 41215
})
Dataset({
    features: ['id', 'question', 'answer', 'messages', 'category', 'complexity', 'turns_count'],
    num_rows: 2168
})


Tokenizer is loaded in the next section, then we apply `tokenizer.apply_chat_template`.

<a id="10-load-qwen3-4b-4-bit-with-unsloth"></a>
## 10. Load Qwen3-4B 4-bit with Unsloth

In [10]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

# Make sure pad token exists.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model and tokenizer loaded.")

==((====))==  Unsloth 2026.6.1: Fast Qwen3 patching. Transformers: 5.10.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.70k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/4.67k [00:00<?, ?B/s]

unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model and tokenizer loaded.


In [11]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

train_text = train_general.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=train_general.column_names,
)

eval_text = test_general.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=test_general.column_names,
)

print(train_text)
print(eval_text)
print("\nSample formatted text:\n")
print(train_text[0]["text"][:1500])

Map:   0%|          | 0/41215 [00:00<?, ? examples/s]

Map:   0%|          | 0/2168 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 41215
})
Dataset({
    features: ['text'],
    num_rows: 2168
})

Sample formatted text:

<|im_start|>user
Theophylline có vai trò thế nào trong điều trị cơn ngừng thở ở trẻ thiếu tháng?<|im_end|>
<|im_start|>assistant
<think>

</think>

**Vai trò của Theophylline trong điều trị cơn ngừng thở ở trẻ thiếu tháng** Theophylline (và dẫn xuất là Aminophylline) có một vị trí quan trọng và được ghi nhận rõ ràng trong điều trị cơn ngừng thở ở trẻ thiếu tháng (Apnea of Prematurity - AOP). Dựa trên các dữ kiện dược lý học và lâm sàng, vai trò này được thể hiện qua các cơ chế sinh bệnh học và đặc điểm dược động học đặc thù sau: ### 1. Cơ chế tác dụng dược lý (Dược lực học) Theophylline tác động lên hệ thần kinh trung ương và hệ hô hấp để ngăn chặn các cơn ngừng thở thông qua các cơ chế sau: * **Kích thích hô hấp trung ương:** Theophylline là chất đối kháng thụ thể Adenosine (Adenosine receptor antagonist). Adenosine là một neuromodulator ức chế hoạt

<a id="11-attach-lora-adapters"></a>
## 11. Attach LoRA Adapters

LoRA trains only a small number of adapter parameters instead of the full model.

In [12]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


<a id="12-training-setup"></a>
## 12. Training Setup

In [13]:
from trl import SFTConfig

# Đưa toàn bộ cấu hình vào trong SFTConfig
sft_config = SFTConfig(
    output_dir = OUTPUT_DIR,
    dataset_text_field = "text",      # Chuyển vào đây
    max_seq_length = MAX_SEQ_LENGTH,  # Chuyển vào đây
    packing = True,                   # Chuyển vào đây
    num_train_epochs = NUM_TRAIN_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    learning_rate = LEARNING_RATE,
    warmup_steps = WARMUP_STEPS,
    logging_steps = LOGGING_STEPS,
    save_steps = SAVE_STEPS,
    save_total_limit = 2,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = SEED,
    report_to = "none",
    eval_strategy = "steps",
    eval_steps = SAVE_STEPS,
)

# SFTTrainer giờ chỉ nhận các model, data và biến args
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_text,
    eval_dataset = eval_text,
    args = sft_config,
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/41215 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/41215 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2168 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=8):   0%|          | 0/2168 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


<a id="13-train-the-model"></a>
## 13. Train the Model

Start training.

In [14]:
import os
# Dán đường dẫn bạn vừa Copy file path ở Bước A vào đây
CHECKPOINT_PATH = "/kaggle/input/datasets/hdtuznn/6000-ckpt/qwen3-4b-medical-qa-lora/checkpoint-6000"

if os.path.exists(CHECKPOINT_PATH):
    print(f"🎉 Đã tìm thấy Checkpoint cũ! Đang khôi phục quá trình huấn luyện từ: {CHECKPOINT_PATH}")
    # Lệnh resume thần thánh
    trainer_stats = trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
    
    # In kết quả sau khi train xong
    print(trainer_stats)
else:
    # Dùng lệnh raise để ép dừng tiến trình ngay lập tức, tránh sinh ra lỗi NameError
    raise FileNotFoundError(f"🚨 Không tìm thấy Checkpoint hợp lệ tại: {CHECKPOINT_PATH}. Đã chủ động ngắt tiến trình để không train lại từ đầu!")

🎉 Đã tìm thấy Checkpoint cũ! Đang khôi phục quá trình huấn luyện từ: /kaggle/input/datasets/hdtuznn/6000-ckpt/qwen3-4b-medical-qa-lora/checkpoint-6000


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 16,662 | Num Epochs = 3 | Total steps = 6,249
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
6249,0.906743,0.953663


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3-4b-medical-qa-lora/checkpoint-6249/tokenizer_config.json.


TrainOutput(global_step=6249, training_loss=0.0355653109636702, metrics={'train_runtime': 9101.5422, 'train_samples_per_second': 5.492, 'train_steps_per_second': 0.687, 'total_flos': 2.237479665925801e+18, 'train_loss': 0.0355653109636702, 'epoch': 3.0})


<a id="14-run-inference-test"></a>
## 14. Run Inference Test

In [15]:
FastLanguageModel.for_inference(model)

def generate_answer(question, max_new_tokens=512):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical question answering assistant. "
                "Answer clearly, cautiously, and remind users to consult healthcare professionals for personal medical decisions."
            ),
        },
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.8,
        do_sample=True,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id,
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return decoded

test_question = "Paracetamol được dùng để làm gì?"
print(generate_answer(test_question))

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


system
You are a medical question answering assistant. Answer clearly, cautiously, and remind users to consult healthcare professionals for personal medical decisions.
user
Paracetamol được dùng để làm gì?
assistant
<think>

</think>

Paracetamol là thuốc giảm đau và hạ sốt phổ biến, được chỉ định chính để điều trị các triệu chứng sau: 1. **Giảm đau**: Các cơn đau nhẹ đến trung bình như đau đầu, đau răng, đau cơ, đau bụng kinh, đau do chấn thương nhỏ, đau sau phẫu thuật (thường trong vòng 24-48 giờ đầu). 2. **Hạ sốt**: Khi nhiệt độ cơ thể tăng cao do nhiễm virus hoặc vi khuẩn. **Lưu ý an toàn quan trọng**: - Chỉ sử dụng khi có chỉ định của bác sĩ hoặc theo hướng dẫn cụ thể. - Liều dùng phụ thuộc vào tuổi, cân nặng và tình trạng bệnh lý. Không vượt quá liều cho phép (ví dụ: người lớn ≤ 4g/ngày, trẻ em cần tính toán chính xác). - Tuyệt đối không tự ý dùng nếu đang dùng thuốc chống đông máu (warfarin), corticoid, hoặc các thuốc khác mà chưa hỏi ý kiến bác sĩ. - Nếu dùng quá liều có thể gâ

<a id="15-save-lora-adapter"></a>
## 15. Save LoRA Adapter

This saves only the LoRA adapter, not the full merged model.

In [16]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved LoRA adapter to:", OUTPUT_DIR)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3-4b-medical-qa-lora/tokenizer_config.json.


Saved LoRA adapter to: /kaggle/working/qwen3-4b-medical-qa-lora


You can download this folder from Kaggle output:

```text
/kaggle/working/qwen3-4b-medical-qa-lora
```

<a id="16-optional-push-adapter-to-hugging-face"></a>
## 16. Optional: Push Adapter to Hugging Face

Only run this section if you have added your Hugging Face token to Kaggle Secrets.

Kaggle Secrets name suggestion:

```text
HF_TOKEN
```

In [17]:
# Optional: login to Hugging Face
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN_WRITE")
login(token=hf_token)

In [18]:
# Optional: push LoRA adapter
HF_REPO_ID = "hdtuann/qwen3-4b-medical-qa-lora"

model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)

print("Pushed to:", HF_REPO_ID)

README.md:   0%|          | 0.00/555 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/hdtuann/qwen3-4b-medical-qa-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpk0ptf240/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Pushed to: hdtuann/qwen3-4b-medical-qa-lora


<a id="17-langgraph-qa-node-skeleton"></a>
## 17. LangGraph QA Node Skeleton

This is not the full final system yet.  
This is the **QA branch node** that will later be connected to:

```text
Classifier Node → Router Node → QA Node / RAG Node → Safety Check Node
```

In [19]:

!pip install -U langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 12.8 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.13
    Uninstalling langgraph-sdk-0.3.13:
      Successfully uninstalled langgraph-sdk-0.3.13
  Attempting uninstall: langgraph-checkpoint
    Found existing installation: langgraph-checkpoint 4.0.1
    Uninstalling langgraph-checkpoint-4.0.1:
      Successfully uninstalled langgraph-checkpoint-4.0.1
  Attempting uninstall: 

In [20]:
# Example LangGraph-compatible state and QA node.
# This will be used later when the RAG branch is ready.

from typing import TypedDict, Optional

class MedicalState(TypedDict):
    question: str
    category: Optional[str]
    route: Optional[str]
    answer: Optional[str]

def qa_chatbot_node(state: MedicalState) -> MedicalState:
    question = state["question"]
    answer = generate_answer(question)

    return {
        **state,
        "route": "general_qa",
        "answer": answer,
    }

# Test node alone
sample_state = {
    "question": "Ibuprofen có tác dụng gì?",
    "category": None,
    "route": None,
    "answer": None,
}

result = qa_chatbot_node(sample_state)
print(result["route"])
print(result["answer"][:1500])

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


general_qa
system
You are a medical question answering assistant. Answer clearly, cautiously, and remind users to consult healthcare professionals for personal medical decisions.
user
Ibuprofen có tác dụng gì?
assistant
<think>

</think>

**CÂU TRẢ LỜI:** Dựa trên thông tin dược lý học và chỉ định lâm sàng, Ibuprofen là thuốc chống viêm không steroid (NSAID) thuộc nhóm acid propionic. Tác dụng chính của Ibuprofen bao gồm: **1. Chống viêm và Giảm đau** - **Cơ chế**: Ibuprofen ức chế enzym Cyclooxygenase (COX), từ đó ngăn chặn quá trình chuyển hóa Acid Arachidonic thành các Prostaglandin (PG). Các Prostaglandin là trung gian gây viêm, đau và sốt. - **Ứng dụng**: Thuốc được dùng để điều trị các cơn đau cấp tính (đau đầu, đau răng, đau bụng...), viêm khớp, và các bệnh lý viêm nhiễm khác. **2. Hạ sốt** - Ibuprofen làm giảm nhiệt độ cơ thể bằng cách ức chế prostaglandin tại vùng dưới đồi (hypothalamus), giúp cân bằng lại điểm đặt nhiệt độ cơ thể. **3. Kháng kết tập tiểu cầu** - Ibuprofen có 

<a id="18-next-step"></a>
## 18. Next Step

After this notebook works, the next notebook should be:

```text
03-build-medical-risk-rag.ipynb
```

That notebook will handle:

```text
safety
interactions
contraindications
contraindication
pregnancy
overdose
pediatric
patient_query
case_based
edge_case
```

Then the final system can be connected with LangGraph:

```text
User Question
      ↓
Classifier
      ↓
Router
   ┌───────────────┬──────────────────┐
   │               │
General QA     Medical Risk RAG
   │               │
   └────── Safety Check ──────┘
              ↓
        Final Answer
```